# Allianz Job Scraper

Web scraper that collects listed jobs from the [Allianz Careers Portal](https://careers.allianz.com/global/en/search-results). Additionally, with the use of selenium applies the Student jobs filter, looking through all pages. Afterwards collects the relevant data and saves it in an excel file with it's date.

## 1.Setup

### 1.1 Installing the Dependencies

In [1]:
!pip install 
!pip install selenium webdriver-manager
!pip install beautifulsoup4
!pip install pandas



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: You must give at least one requirement to install (see "pip help install")



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 1.2 Import Libraries

In [2]:
import math
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


print("All libraries imported successfully!")

All libraries imported successfully!


### 1.3 Configuration

In [3]:
BASE_URL = "https://careers.allianz.com/global/en/search-results"
JOBS_PER_PAGE = 10
SLEEP_BETWEEN_PAGES = 1

## 2 Data Extraction

### 2.1 Job Parsing Function

All jobs lies in a div element. In this function we take this div and look inside of it to get the following data related to the job:
- Title: Lies in an `<a>` element 
- Job Link: Lies in an `<a>` element 
- Location: Get the `<div>` with the following attribute: `data-ph-at-id="job-location"`. Inside of that div element there are two span elements and the exact location lies in the second span. It is important to note that some jobs have more than one location and when that is the case they are inside of a button element and you have to click it to get the value. So when that is the case I saved their location as `"Multiple locations"`
- Category: Lies in a `<div>` element with the following attribute: `data-ph-at-id="job-category"`. Similar to location there are two spans therefore to get the category select the second span. 
- Job Level: Lies in a `<div>` element with the following attribute: `data-ph-at-id="job-jobLevel"`. Similar to location there are two spans therefore to get the category select the second span. This is when you want to search all the jobs otherwise for a specific job level you can omit this.


In [4]:
def parse_job_card(card):
    anchor = card.find("a", attrs={"data-ph-at-id": "job-link"})

    def extract_field(attr_id):
        div = card.find("div", attrs={"data-ph-at-id": attr_id, "role": "text"})
        if div:
            spans = div.find_all("span")
            return spans[1].get_text(strip=True) if len(spans) > 1 else None
        return None

    location = extract_field("job-location")
    if location is None:
        multi_btn = card.find("button", attrs={"data-ph-at-id": "job-multi_location"})
        if multi_btn:
            spans = multi_btn.find_all("span")
            # aria-hidden="true" span at index 1 has the readable text
            location = f"MULTIPLE: {spans[1].get_text(strip=True)}" if len(spans) > 1 else "Multiple locations"

    return {
        "title":     anchor.get_text(strip=True) if anchor else None,
        "link":      anchor.get("href") if anchor else None,
        "location":  location,
        "category":  extract_field("job-category"),
        "job_level": extract_field("job-jobLevel"),
    }

### 2.2 Scraping all Jobs

#### Checking if we get results and if the parsing logic works on the first page before we loop through all pages.

In [5]:
# driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
# wait = WebDriverWait(driver, 15)

# driver.get(f"{BASE_URL}?s=1")

# # Wait for an actual job card to appear rather than just the count span.
# # role="listitem" belongs to the job card divs, so if this element exists
# # we know the cards have finished rendering.
# wait.until(EC.presence_of_element_located(
#     (By.CSS_SELECTOR, "[data-ph-at-id='jobs-list']")
# ))

# soup = BeautifulSoup(driver.page_source, "html.parser")

# total_jobs = int(
#     soup.find("span", attrs={"data-ph-at-id": "search-page-top-job-count"})
#     .get_text(strip=True)
#     .replace(",", "")
# )
# print(f"Total jobs found: {total_jobs}")
# print(f"Total pages to scrape: {math.ceil(total_jobs / JOBS_PER_PAGE)}")

# job_cards = soup.find_all("div", attrs={"data-ph-at-id": "jobs-list"})
# print(f"\nNumber of job cards found on this page: {len(job_cards)}")

# if job_cards:
#     first_job = parse_job_card(job_cards[0])
#     print("\nFirst job parsed result:")
#     for key, value in first_job.items():
#         print(f"  {key}: {value}")
# else:
#     print("\nStill no cards — printing raw snippet to inspect further:")
#     print(soup.body.prettify()[:3000])

# if job_cards:
#     first_job = parse_job_card(job_cards[1])
#     print("\nFirst job parsed result:")
#     for key, value in first_job.items():
#         print(f"  {key}: {value}")
# else:
#     print("\nStill no cards — printing raw snippet to inspect further:")
#     print(soup.body.prettify()[:3000])

# driver.quit()

In [6]:
# job_cards[1]

#### Scraping all the jobs

In [7]:
# all_jobs = []

# driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
# wait = WebDriverWait(driver, 5)

# print("Loading first page...")
# driver.get(f"{BASE_URL}?s=1")

# wait.until(EC.presence_of_element_located(
#     (By.CSS_SELECTOR, "[data-ph-at-id='jobs-list']")
# ))

# soup = BeautifulSoup(driver.page_source, "html.parser")

# total_jobs = int(
#     soup.find("span", attrs={"data-ph-at-id": "search-page-top-job-count"})
#     .get_text(strip=True)
#     .replace(",", "")
# )
# total_pages = math.ceil(total_jobs / JOBS_PER_PAGE)
# print(f"Found {total_jobs} jobs → {total_pages} pages to scrape.")

# for card in soup.find_all("div", attrs={"data-ph-at-id": "jobs-list"}):
#     all_jobs.append(parse_job_card(card))
# print(f"Page 1/{total_pages} done — {len(all_jobs)} jobs so far.")

# for i in range(1, total_pages):
#     from_value = i * JOBS_PER_PAGE
#     driver.get(f"{BASE_URL}?from={from_value}&s=1")
#     wait.until(EC.presence_of_element_located(
#         (By.CSS_SELECTOR, "[data-ph-at-id='jobs-list']")
#     ))

#     soup = BeautifulSoup(driver.page_source, "html.parser")
#     for card in soup.find_all("div", attrs={"data-ph-at-id": "jobs-list"}):
#         all_jobs.append(parse_job_card(card))

#     print(f"Page {i + 1}/{total_pages} done — {len(all_jobs)} jobs so far.")
#     time.sleep(SLEEP_BETWEEN_PAGES)

# driver.quit()
# print(f"\nScraping complete. {len(all_jobs)} total jobs collected.")

In [8]:
# df = pd.DataFrame(all_jobs)


In [9]:
# df.head(), df.info()

In [ ]:
# df.drop_duplicates(subset="link", inplace=True)
# df.to_excel("jobs/allianz_jobs.xlsx", index=False)

# print(f"Saved {len(df)} jobs to jobs/allianz_jobs.xlsx")
# df.head()

### 2.3 Scraping Student Jobs only

Since we are looking for student jobs only we first have to load the page and wait a few second for the contents to appear. After the page is loaded we must first find and click on the student filter (interacting with the checkbox does not work). On the same label we find the student job count which lies between paranthesis. Then we loop through aall of the pages using the `?from=N` URL parameter, collecting 10 jobs per page.

>Important thing to note is that since the student filter is only active on the first page load - navigating by url resets the filter state. Hence location filtering applied post scrape.

In [11]:
all_student_jobs = []

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
wait = WebDriverWait(driver, 15)

print("Loading first page...")
driver.get(f"{BASE_URL}?s=1")
wait.until(EC.presence_of_element_located(
    (By.CSS_SELECTOR, "[data-ph-at-id='jobs-list']")
))

# To search for the student jobs first find the element with the correct label.
# Then click the label to get student jobs
student_label = driver.find_element(By.CSS_SELECTOR, "label[for^='jobLevel_phs_Student']")
driver.execute_script("arguments[0].click();", student_label)

# Wait for the page to reload with filtered results before reading anything
wait.until(EC.presence_of_element_located(
    (By.CSS_SELECTOR, "[data-ph-at-id='jobs-list']")
))

soup = BeautifulSoup(driver.page_source, "html.parser")

student_li = soup.find("li", attrs={"data-ph-at-id": "facet-results-item"}, 
                        string=lambda t: False)

student_input = soup.find("input", attrs={"data-ph-at-text": "Student"})
student_li = student_input.find_parent("li")

count_text = student_li.find("span", attrs={"class": "phw-visually-hidden"}).find_previous_sibling("span").get_text(strip=True)

total_jobs = int(count_text.strip("()"))
total_pages = math.ceil(total_jobs / JOBS_PER_PAGE)
print(f"Found {total_jobs} student jobs → {total_pages} pages to scrape.")


for card in soup.find_all("div", attrs={"data-ph-at-id": "jobs-list"}):
    all_student_jobs.append(parse_job_card(card))
print(f"Page 1/{total_pages} done — {len(all_student_jobs)} jobs so far.")

for i in range(1, total_pages):
    from_value = i * JOBS_PER_PAGE
    driver.get(f"{BASE_URL}?from={from_value}&s=1")
    wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "[data-ph-at-id='jobs-list']")
    ))

    soup = BeautifulSoup(driver.page_source, "html.parser")
    for card in soup.find_all("div", attrs={"data-ph-at-id": "jobs-list"}):
        all_student_jobs.append(parse_job_card(card))

    print(f"Page {i + 1}/{total_pages} done — {len(all_student_jobs)} jobs so far.")
    time.sleep(SLEEP_BETWEEN_PAGES)

driver.quit()
print(f"\nScraping complete. {len(all_student_jobs)} total student jobs collected.")

Loading first page...
Found 182 student jobs → 19 pages to scrape.
Page 1/19 done — 10 jobs so far.
Page 2/19 done — 20 jobs so far.
Page 3/19 done — 30 jobs so far.
Page 4/19 done — 40 jobs so far.
Page 5/19 done — 50 jobs so far.
Page 6/19 done — 60 jobs so far.
Page 7/19 done — 70 jobs so far.
Page 8/19 done — 80 jobs so far.
Page 9/19 done — 90 jobs so far.
Page 10/19 done — 100 jobs so far.
Page 11/19 done — 110 jobs so far.
Page 12/19 done — 120 jobs so far.
Page 13/19 done — 130 jobs so far.
Page 14/19 done — 140 jobs so far.
Page 15/19 done — 150 jobs so far.
Page 16/19 done — 160 jobs so far.
Page 17/19 done — 170 jobs so far.
Page 18/19 done — 180 jobs so far.
Page 19/19 done — 182 jobs so far.

Scraping complete. 182 total student jobs collected.


## 3. Data Processing

### 3.1 Inspecting Results

Loading the data into Pandas Dataframe and running a quick check by printing the dataframe and through `.info()` and `.describe()` functions. 

In [12]:
student_df = pd.DataFrame(all_student_jobs)
student_df, student_df.info(), student_df.describe() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 182 entries, 0 to 181
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   title      182 non-null    object
 1   link       182 non-null    object
 2   location   182 non-null    object
 3   category   182 non-null    object
 4   job_level  182 non-null    object
dtypes: object(5)
memory usage: 7.2+ KB


(                                                 title  \
 0            Graphiste Social Media (H/F) - Alternance   
 1    Product Owner Learning & Development (m/f/d)  ...   
 2                                    Care Centre Agent   
 3    Koordynator/ka ds. Współpracy z Placówkami Med...   
 4    Werkstudent im Bereich Vertriebssteuerung und ...   
 ..                                                 ...   
 177  Working student accounting & billing support (...   
 178  Intern, Institutional Business TW - Limited to...   
 179                Business Development Support Intern   
 180  Working Student Infrastructure Leadership Supp...   
 181                   Estagiário/a Comunicação (m/f/d)   
 
                                                   link  \
 0    https://careers.allianz.com/global/en/job/9791...   
 1    https://careers.allianz.com/global/en/job/9792...   
 2    https://careers.allianz.com/global/en/job/9792...   
 3    https://careers.allianz.com/global/en/job/9792..

### 3.2 Save All Student Jobs

Incase the jobs appeared through multiple pages, we are dropping them by the link, since this is a unique to a single job, then exporting our dataset of all student jobs to an Excel.

In [13]:
student_df.drop_duplicates(subset="link", inplace=True)
student_df.to_excel("./jobs/allianz_student_jobs.xlsx", index=False)

print(f"Saved {len(student_df)} jobs to allianz_student_jobs.xlsx")
student_df.head()

Saved 182 jobs to allianz_student_jobs.xlsx


,title,link,location,category,job_level
0,Graphiste Social Media (H/F) - Alternance,https://careers.allianz.com/global/en/job/9791...,"PUTEAUX, Hauts-de-Seine, France, 92800",Communication & Public Relations,Apprenticeship / Dual Studies
1,Product Owner Learning & Development (m/f/d) ...,https://careers.allianz.com/global/en/job/9792...,"München, Germany, 80802",Human Resources,Professional
2,Care Centre Agent,https://careers.allianz.com/global/en/job/9792...,MULTIPLE: Available in 3 locations,Operations,Entry Level
3,Koordynator/ka ds. Współpracy z Placówkami Med...,https://careers.allianz.com/global/en/job/9792...,"WARSZAWA, Poland, 2-672",Customer Services & Claims,Professional
4,Werkstudent im Bereich Vertriebssteuerung und ...,https://careers.allianz.com/global/en/job/9792...,"Unterföhring (bei München), Germany, 85774",Sales & Distribution,Student


### 3.3 Filter Jobs in Germany

Filtering the dataset so that we only keep the jobs where the location column contains `"Germany"`.

> **Note**: So when doing this you have to consider that you are also dropping jobs whose location column contain `"Multiple locations"`. This can be bad because in the multiple location it might also has Germany as a location. 

In [14]:
student_df_germany = student_df[student_df["location"].str.contains("Germany", case=False, na=False)]
student_df_germany.to_excel("./jobs/allianz_student_jobs_germany.xlsx", index=False)
student_df_germany.head(), student_df_germany.info(), student_df_germany.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 76 entries, 1 to 180
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   title      76 non-null     object
 1   link       76 non-null     object
 2   location   76 non-null     object
 3   category   76 non-null     object
 4   job_level  76 non-null     object
dtypes: object(5)
memory usage: 3.6+ KB


(                                                title  \
 1   Product Owner Learning & Development (m/f/d)  ...   
 4   Werkstudent im Bereich Vertriebssteuerung und ...   
 6   Praktikant im Bereich Rechtsschutz-Schaden (m/...   
 7   Sachbearbeiter Infrastrukturelles Gebäudemanag...   
 11  Werkstudent in Data Products & Agile Transform...   
 
                                                  link  \
 1   https://careers.allianz.com/global/en/job/9792...   
 4   https://careers.allianz.com/global/en/job/9792...   
 6   https://careers.allianz.com/global/en/job/9793...   
 7   https://careers.allianz.com/global/en/job/9794...   
 11  https://careers.allianz.com/global/en/job/9738...   
 
                                       location                    category  \
 1                      München, Germany, 80802             Human Resources   
 4   Unterföhring (bei München), Germany, 85774        Sales & Distribution   
 6   Unterföhring (bei München), Germany, 85774  Customer Servi

### 3.4 Save Jobs in Germany with Date Stamp

In [15]:
import datetime

# get current date in YYYY-MM-DD format to append to the filename
current_date = datetime.datetime.now().strftime("%Y-%m-%d")
current_date

'2026-05-25'

In [16]:
save_to = f"./jobs/jobs_in_germany/allianz_student_jobs_{current_date}.xlsx"
student_df_germany.drop_duplicates(subset="link", inplace=True)

student_df_germany.to_excel(save_to, index=False)

print(f"Saved {len(student_df_germany)} jobs to {save_to}")
student_df_germany.head()

Saved 76 jobs to ./jobs/jobs_in_germany/allianz_student_jobs_2026-05-25.xlsx


C:\Users\rehas\AppData\Local\Temp\ipykernel_12380\865832883.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  student_df_germany.drop_duplicates(subset="link", inplace=True)


,title,link,location,category,job_level
1,Product Owner Learning & Development (m/f/d) ...,https://careers.allianz.com/global/en/job/9792...,"München, Germany, 80802",Human Resources,Professional
4,Werkstudent im Bereich Vertriebssteuerung und ...,https://careers.allianz.com/global/en/job/9792...,"Unterföhring (bei München), Germany, 85774",Sales & Distribution,Student
6,Praktikant im Bereich Rechtsschutz-Schaden (m/...,https://careers.allianz.com/global/en/job/9793...,"Unterföhring (bei München), Germany, 85774",Customer Services & Claims,Student
7,Sachbearbeiter Infrastrukturelles Gebäudemanag...,https://careers.allianz.com/global/en/job/9794...,"Karlsruhe, Germany, 76135",Operations,Professional
11,Werkstudent in Data Products & Agile Transform...,https://careers.allianz.com/global/en/job/9738...,"Unterföhring (bei München), Germany, 85774",Project Management,Student
